In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)


In [ ]:
import plot
from read_chroma import read_only_chroma, read_chromato_and_chromato_cube
import matplotlib.pyplot as plt

from matching import matching_nist_lib_from_chromato_cube
import projection
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import identification


import mass_spec
from sklearn.cluster import DBSCAN
from scipy.signal import savgol_filter
from skimage.restoration import estimate_sigma
from dbscan_peak import detection_mass_par_mass_Dog
import dbscan_peak

In [ ]:
# filename="D:/GCxGC_MS/DATA/Dossier_partagé_GCxGC/Manue/GCxGC_VOLATIL_CF_08bis_postPTR/15-04-25_817822_QC_23newEI.cdf"
# filename = "/home/camille/Documents/app/data/J-A-034-751325-Tedlar.h5"
filename = "/home/camille/Documents/app/data/cdf et h5/A-F-028-817822-droite-ReCIVA.cdf"
filename = "/home/camille/Documents/app/data/cdf et h5/751315_0033CN_J7_postPTR_split5.h5"
chromato_tic, time_rn, chromato_cube, sigma, mass_range=read_chromato_and_chromato_cube(filename, mod_time=1.7,pre_process=False)

In [ ]:

import baseline_correction
method_baseline="als"
baseline_cube = np.array(baseline_correction.chromato_cube_corrected_baseline(chromato_cube, method=method_baseline))

In [ ]:
coordinates, spec_list, area = dbscan_peak.detection_mass_par_mass_Dog(baseline_cube,(chromato_tic, time_rn),
                                                            1.7,
                                                                abs_threshold=500,
                                                                rel_threshold=0.5,
                                                                noise_factor=3,
                                                                min_sigma=1,
                                                                max_sigma=20,
                                                                sigma_ratio=2,
                                                                overlap=0.5, 
                                                                max_peak_per_mass=600,
                                                                rt1_delta=1, 
                                                                rt2_delta=0.01,
                                                                min_size_cluster_mass=3, 
                                                                thr_debscan=0.01, 
                                                                multi_processing=True,
                                                                cleaning_close_peak=True)




In [ ]:
plt.rcParams['figure.figsize'] = [30, 5]
coordinates_in_chromato=projection.matrix_to_chromato(coordinates,  time_rn, 1.7, chromato_tic.shape)
plot.visualizer2((chromato_tic, time_rn), title="chromato", log_chromato=True,  mod_time=1.7,points=coordinates_in_chromato)

In [ ]:
i=0
plot.visualizer2((chromato_tic, time_rn), title="chromato", log_chromato=True,  mod_time=1.7,points=coordinates_in_chromato,
                 rt1=coordinates_in_chromato[i,0],rt2=coordinates_in_chromato[i,1],rt1_window=0.2,rt2_window=1.5)

In [ ]:
import matching
matches = matching.matching_nist_lib_from_chromato_cube(
            (chromato_tic, time_rn, mass_range), baseline_cube, coordinates,
            mod_time=1.7,
            match_factor_min=700, nist=False)
coordinates_in_chromato=projection.matrix_to_chromato(coordinates, time_rn, 1.7, chromato_tic.shape)

In [ ]:
# import h5py
# import netCDF4 as nc
# def get_scan_number(file_path):
#     """Get scan number from file."""
#     try:
#         if file_path.endswith((".h5", ".H5")):
#             with h5py.File(file_path, 'r') as f:
#                 return f.attrs['scan_number_size']
#         elif file_path.endswith((".cdf", ".CDF")):
#             with nc.Dataset(file_path, 'r') as dt:
#                 return dt.dimensions['scan_number'].size
#         else:
#             raise ValueError("Unsupported file format. Please provide a .h5 or .cdf file.")
#     except Exception as e:
#         raise ValueError(f"Error while reading file {file_path}: {e}")

# def get_mod_time(file_path):
#     """Get modulation time based on scan_number from file."""
#     scan_number = get_scan_number(file_path)
#     modulation_times = {
#         328125: (1.25, "G0/plasma"),
#         540035: (1.7, "exhaled air")
#     }
#     if scan_number in modulation_times:
#         mod_time, data_type = modulation_times[scan_number]
#         print(f"   Data type: {data_type}")
#         return mod_time
#     else:
#         print(f"   ⚠️  Unknown scan_number: {scan_number}, using default modulation time")
#         return

In [ ]:
base_name = os.path.splitext(os.path.basename(filename))[0]
formated_spectra=True
quant ="masse"
extract_patch=False
output_hdf5_file=None

In [ ]:

def visualize_integration_methods(chromato_m, coord,  peak_name="Peak",  SNR=3):
    """
    Visualise les deux méthodes d'intégration sur le même graphique
    
    Parameters:
    -----------
    chromato_m : ndarray
        Chromatogramme 2D
    coord : tuple
        (rt1_idx, rt2_idx) position du pic
    mod_time : float
        Temps de modulation
    FWHM : float
        Largeur à mi-hauteur 
    peak_name : str
        Nom du pic pour le titre
    """
    
    # Extraire le profil RT2
    profile = chromato_m[coord[0], :]
    peak_idx = coord[1]
    
    # Paramètres ChromaTOF
    # rt2_hz = len(profile) / mod_time  # ou chromato.shape[1] / mod_time
    # baseline_width_sec = FWHM * 2.35
    # expected_width = int(baseline_width_sec * rt2_hz)
    
    # === MÉTHODE 1: ChromaTOF ===
    snr = SNR
    left_chr, right_chr = identification.method_chromatof(profile, peak_idx, snr)
    area_chromatof = np.sum(profile[left_chr:right_chr+1])
    
    # === MÉTHODE 2: Robuste ===
    left_rob, right_rob = identification.find_peak_bounds_intelligent(profile, peak_idx)
    # left_rob, right_rob = find_peak_bounds_robust(profile, peak_idx, fwhm_factor=FWHM_FACTOR)
    area_robust = np.sum(profile[left_rob:right_rob+1])
    
    # === VISUALISATION ===
    plt.figure(figsize=(14, 8))
    
    # Graphique principal
    plt.subplot(2, 2, (1, 2))  # Large plot en haut
    
    # Profil complet
    x_axis = np.arange(len(profile))
    plt.plot(x_axis, profile, 'b-', linewidth=2, label='Profil RT2', alpha=0.7)
    
    # Peak apex
    plt.scatter([peak_idx], [profile[peak_idx]], color='black', s=100, 
                marker='x', linewidth=3, label=f'Apex (RT2={peak_idx})')
    
    # Bornes ChromaTOF
    plt.axvline(left_chr, color='red', linestyle='--', linewidth=2, alpha=0.8)
    plt.axvline(right_chr, color='red', linestyle='--', linewidth=2, alpha=0.8)
    plt.fill_between(x_axis[left_chr:right_chr+1], 
                     profile[left_chr:right_chr+1], 
                     alpha=0.3, color='red', label=f'ChromaTOF: [{left_chr}:{right_chr}]')
    
    # Bornes Robuste
    plt.axvline(left_rob, color='green', linestyle=':', linewidth=2, alpha=0.8)
    plt.axvline(right_rob, color='green', linestyle=':', linewidth=2, alpha=0.8)
    plt.fill_between(x_axis[left_rob:right_rob+1], 
                     profile[left_rob:right_rob+1], 
                     alpha=0.2, color='green', label=f'Robuste: [{left_rob}:{right_rob}]')
    
    plt.title(f'{peak_name} - Comparaison des Méthodes d\'Intégration\n'
              f'Coord: {coord} ')
    plt.xlabel('RT2 Index')
    plt.ylabel('Intensité')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # === ZOOM SUR LE PIC ===
    plt.subplot(2, 2, 3)
    
    # Définir zone de zoom
    zoom_left = max(0, min(left_chr, left_rob) - 5)
    zoom_right = min(len(profile), max(right_chr, right_rob) + 5)
    zoom_x = x_axis[zoom_left:zoom_right]
    zoom_profile = profile[zoom_left:zoom_right]
    
    plt.plot(zoom_x, zoom_profile, 'b-', linewidth=2)
    plt.scatter([peak_idx], [profile[peak_idx]], color='black', s=80, marker='x')
    
    # Bornes dans le zoom
    plt.axvline(left_chr, color='red', linestyle='--', alpha=0.8)
    plt.axvline(right_chr, color='red', linestyle='--', alpha=0.8)
    plt.axvline(left_rob, color='green', linestyle=':', alpha=0.8)
    plt.axvline(right_rob, color='green', linestyle=':', alpha=0.8)
    
    plt.title('Zoom sur le Pic')
    plt.xlabel('RT2 Index')
    plt.ylabel('Intensité')
    plt.grid(True, alpha=0.3)
    
    # === STATISTIQUES ===
    plt.subplot(2, 2, 4)
    plt.axis('off')
    
    # Calculs comparatifs
    width_chr = right_chr - left_chr + 1
    width_rob = right_rob - left_rob + 1
    diff_area = abs(area_chromatof - area_robust)
    diff_percent = (diff_area / max(area_chromatof, area_robust)) * 100
    
    stats_text = f"""
📊 STATISTIQUES COMPARATIVES

🔴 ChromaTOF:
   Bornes: [{left_chr}:{right_chr}]
   Largeur: {width_chr} points
   Aire: {area_chromatof:,.0f}
   SNR: {snr:.1f}


🟢 Robuste:
   Bornes: [{left_rob}:{right_rob}]
   Largeur: {width_rob} points  
   Aire: {area_robust:,.0f}


📈 Différences:
   Δ Aire: {diff_area:,.0f} ({diff_percent:.1f}%)
   Δ Largeur: {width_chr - width_rob:+d} points


"""
    
    plt.text(0.1, 0.9, stats_text, transform=plt.gca().transAxes, 
             fontsize=10, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout()
    plt.show()
    
    # Retourner les résultats pour usage ultérieur
    return {
        'chromatof': {'bounds': (left_chr, right_chr), 'area': area_chromatof},
        'robust': {'bounds': (left_rob, right_rob), 'area': area_robust},
        # 'parameters': {'rt2_hz': rt2_hz, 'expected_width': expected_width}
    }

In [ ]:
def compute_matches_identification(matches, sepc_list, area,chromato, chromato_cube,time_rn,
                                   mass_range, sample_name, formated_spectra,
                                   quant="mass",extract_patch=False,output_hdf5_file=None,
                                   ):
    # print("compute matches , chromatocube shape:", chromato_cube.shape)
    matches_identification = []
    sample_metadata_list =[]
    max_len = max(len(match) for match in matches)

    # ComplÃ©ter les lignes plus courtes ou tronquer les lignes trop longues
    matches = [match + [None] * (max_len - len(match)) if len(match) < max_len else match[:max_len] for match in matches]
    matches = np.array(matches, dtype=object)
    min_mz, max_mz = int(mass_range[0]), int(mass_range[1])
    canonical_length = max_mz - min_mz + 1
    basename = os.path.splitext(sample_name)[0]
    sample_name_group = basename
    if extract_patch: 
        with h5py.File(output_hdf5_file, "a") as h5_file:
            if sample_name_group not in h5_file:
                sample_group_h5 = h5_file.create_group(sample_name_group)

    for j, match in enumerate(matches):
        
        match_data_list = match[1] \
            if isinstance(match[1], list) else [match[1]]
        
        coord = match[2]
        spectrum_data = match[1][0]['spectra']
        spec_deconvo = sepc_list[j]
        majority_mass = np.argmax(spec_deconvo)
        
        if(quant == "mass"):
            chromato_m = chromato_cube[majority_mass,: ,: ] ## pas sur le chromato mais sur la masse majoritaire 
        else: 
            chromato_m=chromato
        
        # TODO tester
        # FWHM = 0.04
        # FWHM_FACTOR = 0.85
        SNR = 3
        if j < 20:  # Afficher seulement les 10 premiers pics
            results = visualize_integration_methods(
                chromato_m, coord,  peak_name=f"Peak {j+1}", SNR=SNR
            )

        

In [ ]:
matches_identification, sample_metadata_list = compute_matches_identification(
                matches, spec_list, area, chromato_tic, baseline_cube,time_rn, mass_range,base_name,
                formated_spectra, quant, extract_patch, output_hdf5_file)

